# Run the whole pipeline

One entry point for a fresh workspace: Bronze -> Silver -> Gold -> Validation -> Monitoring.

| Parameter | Default | Meaning |
|---|---|---|
| `catalog` | `workspace` | Unity Catalog catalog where `bda_bronze`, `bda_silver`, `bda_gold` are created |
| `source_schema` | `samples.tpch` | where the (pre-production) TPC-H source tables live |

Every child notebook receives the same parameters, so the pipeline can be pointed at another workspace or a pre-production copy without code changes. Critical validation failures raise and stop the run. The same notebook can be used as the single task of a Databricks Job.

In [ ]:
# Portability: target catalog and source data location are parameters.
# Defaults reproduce the original setup (Databricks Free Edition); override them via widgets or job parameters.
import re
dbutils.widgets.text("catalog", "workspace", "Target catalog")
dbutils.widgets.text("source_schema", "samples.tpch", "Source data (catalog.schema)")
CATALOG = dbutils.widgets.get("catalog").strip()
SOURCE_SCHEMA = dbutils.widgets.get("source_schema").strip()
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG):
    raise ValueError(f"invalid catalog: {CATALOG!r}")
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*(\.[A-Za-z_][A-Za-z0-9_]*)?", SOURCE_SCHEMA):
    raise ValueError(f"invalid source_schema: {SOURCE_SCHEMA!r}")
spark.sql(f"USE CATALOG {CATALOG}")
print(f"catalog={CATALOG}  source_schema={SOURCE_SCHEMA}")

In [ ]:
import time

ARGS = {"catalog": CATALOG, "source_schema": SOURCE_SCHEMA}
STEPS = [
    ("./00_bronze_layer_ingestion", ARGS),
    ("./01_silver_layer", ARGS),
    ("./05_run_member2", ARGS),  # 02 Gold -> 04 Monitoring (runs 03 Validation twice)
]
for path, args in STEPS:
    t0 = time.time()
    print(f"-> {path}")
    dbutils.notebook.run(path, 0, args)
    print(f"   done in {time.time() - t0:,.0f}s")
print("pipeline finished: Gold is validated; open 07_business_dashboard or the Databricks dashboard")